### Rag Pipelines - Data Ingestion -> Vector DB Pipeline

In [21]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
import os

In [22]:
### Reading all the pdf's from the directory

def Process_all_pdfs(pdf_directory):
    """Process all the pdf's from the directory"""
    all_docs = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_docs.extend(documents)
            print(f"✔️ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"❌ Error : {e}")
    print(f"\nTotal documents loaded: {len(all_docs)}")
    return all_docs

all_pdf_documents = Process_all_pdfs("../data/")

Found 4 PDF files to process.

Processing: ..\data\pdf\Attention.pdf
✔️ Loaded 15 pages

Processing: ..\data\pdf\embedding.pdf
✔️ Loaded 14 pages

Processing: ..\data\pdf\objectdetection.pdf
✔️ Loaded 21 pages

Processing: ..\data\pdf\proposal.pdf
✔️ Loaded 8 pages

Total documents loaded: 58


In [23]:
all_pdf_documents

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [24]:
### text splitting getting into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split  {len(documents)}documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"Example chunk:")
        print(f"content : {split_docs[0].page_content[:200]}...")
        print(f"Metadata : {split_docs[0].metadata}")   
    return split_docs
chunks = split_documents(all_pdf_documents)

Split  58documents into 358 chunks.
Example chunk:
content : Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
...
Metadata : {'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}


### Embedding and VectorStore DB

In [25]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [26]:
class EmbeddingManager:
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the Embedding Manager.

        Args:
        model_name : HuggingFace modelname for sentence embeddings 
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
           print(f"Loading model '{self.model_name}'...")
           self.model = SentenceTransformer(self.model_name)
           print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise

    def genrate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts : List of texts to  embed 
        Returns:
            numpy array of embeddings with the shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initilize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3718.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [27]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 716


In [28]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\Attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'source_file': 'Attention.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of T

In [29]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Genrate the Embeddings

embeddings = embedding_manager.genrate_embeddings(texts)

## store it in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 358 texts...


Batches: 100%|██████████| 12/12 [00:18<00:00,  1.57s/it]


Generated embeddings with shape: (358, 384)
Adding 358 documents to vector store...
Successfully added 358 documents to vector store
Total documents in collection: 1074


### Retriever pipeline from vector store

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager = EmbeddingManager):
        """
        Initialize the retriever
        Args:
            vector_store = Vector store containing document embeddings
            embedding_manager : Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
        
    def retrieve(self, query: str, top_k: int=5, score_threshold: float=0.0 ) -> List[Dict[str,Any]]:
        """
        Retrieve relevant documents for a query
        Args:
            query : The search query
            top_k : Number of top results to retrun 
            score_threshold : Minimum simalarity score threshold
        Returns:
            List of dictionaries containing retrieved docs and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, score_threshold: {score_threshold}")

        # Genrate the embedding query
        query_embedding = self.embedding_manager.genrate_embeddings([query])[0]

        # search in vector Store
        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            # process the results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id,document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to to similarity score (ChromaDB uses the cosine distance)
                    simailarity_score= 1- distance

                    if simailarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id":doc_id,
                            "content":document,
                            "metadata": metadata,
                            "similarity_score":simailarity_score,
                            "distance":distance,
                            "rank":i+1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during reterival: {e}")
            return []
        
rag_retriever = RAGRetriever(vectorstore, embedding_manager)


In [31]:
rag_retriever.retrieve("OOV (out-of-vocabulary) numbers challenge embedding models. ")

Retrieving documents for query: 'OOV (out-of-vocabulary) numbers challenge embedding models. '
Top K: 5, score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 37.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_31a82d3a_83',
  'content': 'culty of numeracy compared to literacy and again\nhighlights the need for specialized designs directly\ntargeting numeracy.\nOOV (out-of-vocabulary) numbers challenge\nembedding models.Analyzing the results based\non number formatting, we find that embedding\nmodels consistently struggle with decimal num-\nbers (e.g., 8.67), large integers (e.g., 3035), and\ncomma-separated numbers (e.g., 3,035). A closer\nlook reveals that numbers in these formats are of-\nten out-of-vocabulary (OOV), meaning they are\nmore likely to be split into multiple smaller units\n(tokens) during tokenization. For example, in a\ncomma-separated number, the comma often marks\na boundary, leading to a number like “1,100” be-\ning broken into “1”, “,”, and “100”.\nThis could be problematic because splitting a\nnumber can disrupt key information, such as its\nmagnitude, which is essential for many down-\nstream tasks. Thus, we believe addressing this\nOOV issue has great pot

### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
# load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key =  "your api key"
llm=ChatGroq(groq_api_key=groq_api_key,model_name="qwen/qwen3-32b",temperature=0.1,max_tokens=1024)


## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:
        """
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [44]:
answer=rag_simple("What is attention mechanism?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'What is attention mechanism?'
Top K: 3, score_threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 34.42it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


<think>
Okay, the user is asking about the attention mechanism. Let me look at the context provided. The context is from section 3.2 Attention, which mentions that an attention function maps a query and key-value pairs to an output. All these elements—query, keys, values, output—are vectors. The output is a weighted sum.

Hmm, the context is repeated three times, but the key points are there. The attention mechanism is about computing a weighted sum based on the query and keys. The weights are determined by the similarity between the query and each key. Then, these weights are applied to the values to get the output. 

Wait, the user wants a concise answer. So I need to summarize that the attention mechanism uses queries, keys, and values to compute a weighted sum where the weights are based on the relevance (similarity) of the query to each key. The output is this sum, which helps the model focus on important parts of the input. 

I should make sure not to add extra information beyond

### Enhanced RAG Pipeline Features

In [ ]:
# --------- Enhanced RAG pipeline Feature -----------

def rag_advanced(query, retriever, llm, top_k = 5, min_score=0.2, return_context = False ):
    """
    RAG pipeline with extra features:
    - Returns answer, sourced, confiedence score, and optionally full context.
    """

    results = retriever.retrieve(query, top_k = top_k, score_threshold = min_score)

    if not results:
        return {"answer":"No relevant context found.","sources":[],"confidence":0.0, "context":''}
    
    # prepare context and sources

    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata' ].get('source_file', doc['metadata'].get('source', 'unknown' )),
        'page' : doc['metadata' ].get('page', 'unknown' ),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] +"..."
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # genrate answer
    prompt = f"""Use the following context to answer the question concisely.
            \nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("Does Seeing Numbers More Often Help Models Handle Them Better?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Retrieving documents for query: 'Does Seeing Numbers More Often Help Models Handle Them Better?'
Top K: 3, score_threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.66it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: <think>
Okay, let's tackle this question. The user is asking if seeing numbers more often helps models handle them better. The context provided talks about how number precision and significant figures affect model performance. 

First, I need to parse the key points from the context. The main idea is that the number of significant figures (s.f.) in numbers impacts model accuracy. As the number of s.f. increases, model performance decreases. They mention that numbers like 0.8 (which has one s.f.) have higher accuracy compared to numbers with more s.f. like 0.0006. 

The context also draws a parallel to human cognition, where more significant figures mean higher cognitive load and more errors. So, models might be struggling similarly when numbers have more precision.

The question is about frequency of seeing numbers. The context doesn't directly mention frequency. Instead, it's about the inherent complexity of the numbers (precision/s.f.). However, the user might be implying tha

In [52]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'what is attention is all you need'
Top K: 3, score_threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 36.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention functi

on can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

3.2 Attention
An attention function can be described as mapping a query and a set of key-value pairs to an output,
where the query, keys, values, and output are all vectors. The output is computed as a weighted sum
3

Question: what is attention is all you need

Answer:

Final Answer: <think>
Okay, the user is asking "what is attention is all you need." Let me start by recalling the context provided. The context is about the attention function, which maps a query and key-value pairs to an output using a weighted sum. The repetition of section 3.2 suggests it's from a paper, probably the original Transformer paper titled "Attention Is All You Need."

So, the answer should connect the attention mechanism described in the context to the Transformer model. The key points are that attention functions use quer